In [1]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)
from authlib.integrations.base_client.errors import OAuthError

Python executable:
C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Scripts\python.exe

Python version:
3.14.10 (tags/v3.14.10:b703b2c, Oct 25 2026, 17:36:52) [MSC v.1944 64 bit (AMD64)]


In [2]:
import os
import secrets
import sys
import jwt
import requests

from getpass import getpass
from flask import Flask, redirect, session, url_for, jsonify
from authlib.integrations.flask_client import OAuth
from urllib.parse import urlencode
from werkzeug.serving import make_server
from threading import Thread
from jwt import PyJWKClient
from html import escape

print("Python:", sys.executable)
print("Libraries imported successfully!")

Python: C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Scripts\python.exe
Libraries imported successfully!


In [3]:
AUTH0_DOMAIN = input("Enter Auth0 Domain: ").strip()
AUTH0_DOMAIN = AUTH0_DOMAIN.replace("https://", "").rstrip("/")

AUTH0_CLIENT_ID = input("Enter Regular Web App Client ID: ").strip()
AUTH0_CLIENT_SECRET = getpass("Enter Regular Web App Client Secret: ")

AUTH0_API_AUDIENCE = "https://api.securenova.com"

FLASK_SECRET_KEY = secrets.token_urlsafe(32)

print("\nConfiguration loaded successfully!")
print("Auth0 Domain:", AUTH0_DOMAIN)
print("API Audience:", AUTH0_API_AUDIENCE)
print("Client Secret: Hidden")

Enter Auth0 Domain:  dev-wuaza68d7rbydra8.us.auth0.com
Enter Regular Web App Client ID:  2Ae7PwrFePk9PI075AQATAcPda0lvce3
Enter Regular Web App Client Secret:  ········



Configuration loaded successfully!
Auth0 Domain: dev-wuaza68d7rbydra8.us.auth0.com
API Audience: https://api.securenova.com
Client Secret: Hidden


In [4]:
app = Flask(__name__)

app.secret_key = FLASK_SECRET_KEY

app.config.update(
    SESSION_COOKIE_HTTPONLY=True,
    SESSION_COOKIE_SAMESITE="Lax",
    SESSION_COOKIE_SECURE=False
)

oauth = OAuth(app)

oauth.register(
    name="auth0",
    client_id=AUTH0_CLIENT_ID,
    client_secret=AUTH0_CLIENT_SECRET,
    server_metadata_url=(
        f"https://{AUTH0_DOMAIN}/.well-known/openid-configuration"
    ),
    client_kwargs={
        "scope": "openid profile email read:ai-data offline_access",
        "code_challenge_method": "S256"
    }
)

print("Flask application initialized!")
print("Auth0 OAuth configuration completed!")

Flask application initialized!
Auth0 OAuth configuration completed!


In [5]:
token_store = {}
id_token_store = {}

latest_token_session_id = None

jwks_client = PyJWKClient(
    f"https://{AUTH0_DOMAIN}/.well-known/jwks.json"
)

print("Token storage initialized!")
print("JWT verification configured!")

Token storage initialized!
JWT verification configured!


In [6]:
@app.route("/")
def home():

    if session.get("user"):
        return """
        <h1>Welcome to SecureNova AI</h1>
        <p>You are logged in successfully.</p>
        <a href="/dashboard">Open Dashboard</a><br><br>
        <a href="/logout">Logout</a>
        """

    return """
    <h1>Welcome to SecureNova AI</h1>
    <p>Secure AI Chat Application</p>
    <a href="/login">Login with Auth0</a>
    """


@app.route("/login")
def login():

    redirect_uri = "http://localhost:5003/callback"

    return oauth.auth0.authorize_redirect(
        redirect_uri,
        audience=AUTH0_API_AUDIENCE
    )


print("Homepage and login routes registered!")

Homepage and login routes registered!


In [7]:

@app.route("/callback")
def callback():

    app.logger.warning("CALLBACK DIAGNOSTIC: UPDATED CODE IS RUNNING")

    global latest_token_session_id

    token = oauth.auth0.authorize_access_token()

    app.logger.warning(
        "CALLBACK DIAGNOSTIC: TOKEN KEYS = %s",
        list(token.keys())
    )

    print("Token keys received:", list(token.keys()))

    user_info = token.get("userinfo")

    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    id_token = token.get("id_token")
    access_token = token.get("access_token")
    refresh_token = token.get("refresh_token")

    if not id_token or not access_token:
        raise RuntimeError("ID token or access token is missing.")

    token_session_id = secrets.token_urlsafe(32)

    token_store[token_session_id] = access_token
    id_token_store[token_session_id] = id_token

    if refresh_token:
        token_store[f"{token_session_id}_refresh"] = refresh_token
        print("Refresh token received successfully.")
    else:
        print("Refresh token was not returned.")

    latest_token_session_id = token_session_id

    session["token_session_id"] = token_session_id

    session["user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "email": user_info.get("email", ""),
        "sub": user_info.get("sub", "")
    }

    return redirect(url_for("dashboard"))


print("Callback route registered!")


Callback route registered!


In [8]:
@app.route("/dashboard")
def dashboard():

    user = session.get("user")

    if not user:
        return redirect(url_for("login"))

    name = escape(user["name"])
    email = escape(user["email"])
    subject = escape(user["sub"])

    return f"""
    <h1>SecureNova AI Dashboard</h1>
    <h3>Welcome, {name}</h3>
    <p>Email: {email}</p>
    <p>Auth0 User ID: {subject}</p>

    <hr>

    <a href="/api/user-data">Test User API</a><br><br>
    <a href="/api/admin">Test Admin API</a><br><br>
    <a href="/logout">Logout</a>
    """


@app.route("/logout")
def logout():

    token_session_id = session.get("token_session_id")

    if token_session_id:
        token_store.pop(token_session_id, None)
        id_token_store.pop(token_session_id, None)

    session.clear()

    logout_url = (
        f"https://{AUTH0_DOMAIN}/v2/logout?"
        + urlencode({
            "client_id": AUTH0_CLIENT_ID,
            "returnTo": "http://localhost:5003/"
        })
    )

    return redirect(logout_url)


print("Dashboard and logout routes registered!")

Dashboard and logout routes registered!


In [9]:
def verify_current_user_token():

    token_session_id = session.get("token_session_id")

    if not token_session_id:
        return None

    access_token = token_store.get(token_session_id)

    if not access_token:
        return None

    try:
        signing_key = jwks_client.get_signing_key_from_jwt(
            access_token
        )

        claims = jwt.decode(
            access_token,
            signing_key.key,
            algorithms=["RS256"],
            audience=AUTH0_API_AUDIENCE,
            issuer=f"https://{AUTH0_DOMAIN}/"
        )

        return claims

    except jwt.InvalidTokenError:
        return None

    except Exception as error:
        print("JWT verification error:", type(error).__name__)
        return None


print("JWT verification function configured!")

JWT verification function configured!


In [10]:
@app.route("/api/user-data")
def api_user_data():

    claims = verify_current_user_token()

    if claims is None:
        return jsonify({
            "error": "Invalid or expired access token"
        }), 401

    scopes = claims.get("scope", "").split()

    if "read:ai-data" not in scopes:
        return jsonify({
            "error": "Insufficient permissions"
        }), 403

    return jsonify({
        "message": "User data accessed successfully!",
        "scope": scopes,
        "access": "READ"
    }), 200


@app.route("/api/admin")
def api_admin():

    claims = verify_current_user_token()

    if claims is None:
        return jsonify({
            "error": "Invalid or expired access token"
        }), 401

    scopes = claims.get("scope", "").split()

    if "write:admin" not in scopes:
        return jsonify({
            "error": "Forbidden: Admin permission required"
        }), 403

    return jsonify({
        "message": "Admin access granted!",
        "access": "ADMIN"
    }), 200


print("Protected API endpoints registered!")

Protected API endpoints registered!


In [12]:
server = make_server("127.0.0.1", 5003, app)

server_thread = Thread(
    target=server.serve_forever,
    daemon=True
)

server_thread.start()

print("SecureNova AI server started successfully!")
print("Open: http://localhost:5003")

SecureNova AI server started successfully!
Open: http://localhost:5003


127.0.0.1 - - [11/Nov/2026 12:01:13] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Nov/2026 12:01:14] "GET /login HTTP/1.1" 302 -
[2026-11-11 12:01:46,467] WARNING in 3272895564: CALLBACK DIAGNOSTIC: UPDATED CODE IS RUNNING
[2026-11-11 12:01:47,640] WARNING in 3272895564: CALLBACK DIAGNOSTIC: TOKEN KEYS = ['access_token', 'refresh_token', 'id_token', 'scope', 'expires_in', 'token_type', 'expires_at', 'userinfo']
127.0.0.1 - - [11/Nov/2026 12:01:47] "GET /callback?code=PqLwz9iuXHT4JOVeFGeNN2ecN8_xrKgWjnnGsmFRwpstK&state=LPeaXQXX6vSkgkdZCsyVikfNf49BQx&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [11/Nov/2026 12:01:47] "GET /dashboard HTTP/1.1" 200 -


Token keys received: ['access_token', 'refresh_token', 'id_token', 'scope', 'expires_in', 'token_type', 'expires_at', 'userinfo']
Refresh token received successfully.


127.0.0.1 - - [11/Nov/2026 12:02:54] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [11/Nov/2026 12:02:55] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Nov/2026 12:02:56] "GET /login HTTP/1.1" 302 -
[2026-11-11 12:03:20,560] WARNING in 3272895564: CALLBACK DIAGNOSTIC: UPDATED CODE IS RUNNING
[2026-11-11 12:03:20,564] ERROR in app: Exception on /callback [GET]
Traceback (most recent call last):
  File "C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.

In [13]:
server.shutdown()
server_thread.join(timeout=5)

print("Flask server stopped successfully!")

Flask server stopped successfully!


In [13]:
refresh_key = f"{latest_token_session_id}_refresh"

print("Latest session exists:", latest_token_session_id is not None)
print("Refresh token stored:", refresh_key in token_store)
print("Auth0 client ID configured:", bool(oauth.auth0.client_id))
print("Auth0 client secret configured:", bool(oauth.auth0.client_secret))

Latest session exists: True
Refresh token stored: True
Auth0 client ID configured: True
Auth0 client secret configured: True


In [14]:
#Step 9 – Exchange the original refresh token

import requests

# Get the refresh token from the latest login session
refresh_key = f"{latest_token_session_id}_refresh"
old_refresh_token = token_store.get(refresh_key)

if not old_refresh_token:
    raise RuntimeError("Refresh token not found. Please log in again.")

# Keep the original token for the replay test
rotation_test_old_refresh_token = old_refresh_token

# Get Auth0 issuer information
issuer = oauth.auth0.server_metadata.get("issuer")

if not issuer:
    raise RuntimeError("Auth0 issuer information is unavailable.")

token_endpoint = issuer.rstrip("/") + "/oauth/token"

# Prepare refresh-token request
payload = {
    "grant_type": "refresh_token",
    "client_id": oauth.auth0.client_id,
    "refresh_token": old_refresh_token
}

if oauth.auth0.client_secret:
    payload["client_secret"] = oauth.auth0.client_secret

# Exchange original refresh token
response = requests.post(
    token_endpoint,
    data=payload,
    timeout=30
)

print("HTTP status:", response.status_code)

result = response.json()

if response.status_code == 200:
    new_refresh_token = result.get("refresh_token")

    if new_refresh_token:
        token_store[refresh_key] = new_refresh_token
        print("Refresh token rotation successful!")
        print("New refresh token received and stored.")
        print("Original refresh token preserved for replay testing.")
    else:
        print("No replacement refresh token returned.")
        print("Stop here and check the Auth0 rotation configuration.")
else:
    print("Refresh token exchange failed.")
    print("Error:", result.get("error"))
    print("Description:", result.get("error_description"))


HTTP status: 200
Refresh token rotation successful!
New refresh token received and stored.
Original refresh token preserved for replay testing.


In [16]:
#Step 11 – Replay the original refresh token

# Replay the original refresh token
replay_payload = {
    "grant_type": "refresh_token",
    "client_id": oauth.auth0.client_id,
    "refresh_token": rotation_test_old_refresh_token
}

if oauth.auth0.client_secret:
    replay_payload["client_secret"] = oauth.auth0.client_secret

replay_response = requests.post(
    token_endpoint,
    data=replay_payload,
    timeout=30
)

print("Replay HTTP status:", replay_response.status_code)

replay_result = replay_response.json()

if replay_response.status_code != 200:
    print("Original refresh token rejected.")
    print("Error:", replay_result.get("error"))
    print("Description:", replay_result.get("error_description"))
else:
    print("WARNING: Original refresh token was accepted.")


Replay HTTP status: 403
Original refresh token rejected.
Error: invalid_grant
Description: Unknown or invalid refresh token.


In [109]:
import inspect

registered_callback = app.view_functions.get("callback")

print("Callback registered:", registered_callback is not None)

if registered_callback:
    print("\nActual callback code used by Flask:\n")
    print(inspect.getsource(registered_callback))

Callback registered: True

Actual callback code used by Flask:

@app.route("/callback")
def callback():

    app.logger.warning("CALLBACK DIAGNOSTIC: UPDATED CODE IS RUNNING")

    global latest_token_session_id

    token = oauth.auth0.authorize_access_token()

    app.logger.warning(
        "CALLBACK DIAGNOSTIC: TOKEN KEYS = %s",
        list(token.keys())
    )

    print("Token keys received:", list(token.keys()))

    user_info = token.get("userinfo")

    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    id_token = token.get("id_token")
    access_token = token.get("access_token")
    refresh_token = token.get("refresh_token")

    if not id_token or not access_token:
        raise RuntimeError("ID token or access token is missing.")

    token_session_id = secrets.token_urlsafe(32)

    token_store[token_session_id] = access_token
    id_token_store[token_session_id] = id_token

    if refresh_token:
        token_store[f"{token_session

In [108]:
print("Latest session ID exists:",
      latest_token_session_id is not None)

print("ID tokens stored:", len(id_token_store))
print("Access tokens stored:", len(token_store))

Latest session ID exists: True
ID tokens stored: 1
Access tokens stored: 1


127.0.0.1 - - [11/Nov/2026 11:12:25] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Nov/2026 11:12:26] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [11/Nov/2026 11:12:30] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Nov/2026 11:12:33] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [11/Nov/2026 11:12:57] "GET /callback?code=tDIXuUO_V1-pWSwfgGrdoGcVuLaFaf08j2N4IbLie9S7n&state=svAYVlWoOp3yqzwd8bjFF96Bi3ifg9&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [11/Nov/2026 11:12:57] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [11/Nov/2026 11:14:04] "GET / HTTP/1.1" 200 -


In [61]:
id_token = id_token_store[latest_token_session_id]

signing_key = jwks_client.get_signing_key_from_jwt(id_token)

id_claims = jwt.decode(
    id_token,
    signing_key.key,
    algorithms=["RS256"],
    audience=AUTH0_CLIENT_ID,
    issuer=f"https://{AUTH0_DOMAIN}/"
)

agent_claim = "https://securenova.com/claims/agent_id"

print("ID token signature verified!")
print("Issuer:", id_claims["iss"])
print("Audience:", id_claims["aud"])
print("Agent ID:", id_claims.get(agent_claim))

ID token signature verified!
Issuer: https://dev-wuaza68d7rbydra8.us.auth0.com/
Audience: 2Ae7PwrFePk9PI075AQATAcPda0lvce3
Agent ID: securenova-ai-agent-01


In [62]:
from getpass import getpass

SSO_CLIENT_ID = input(
    "Enter SSO Demo Client ID: "
).strip()

SSO_CLIENT_SECRET = getpass(
    "Enter SSO Demo Client Secret: "
)

print("\nSSO configuration loaded successfully!")
print("Client ID loaded:", bool(SSO_CLIENT_ID))
print("Client Secret loaded:", bool(SSO_CLIENT_SECRET))

Enter SSO Demo Client ID:  qpgg1cYorZqgFQ9aw1J2URxWbBP0zKsF
Enter SSO Demo Client Secret:  ········



SSO configuration loaded successfully!
Client ID loaded: True
Client Secret loaded: True


In [15]:
sso_app = Flask("SecureNovaSSODemo")

sso_app.secret_key = secrets.token_urlsafe(32)

sso_app.config.update(
    SESSION_COOKIE_NAME="securenova_sso_session",
    SESSION_COOKIE_HTTPONLY=True,
    SESSION_COOKIE_SAMESITE="Lax",
    SESSION_COOKIE_SECURE=False
)

sso_oauth = OAuth(sso_app)

sso_oauth.register(
    name="auth0",
    client_id=SSO_CLIENT_ID,
    client_secret=SSO_CLIENT_SECRET,
    server_metadata_url=(
        f"https://{AUTH0_DOMAIN}/.well-known/openid-configuration"
    ),
    client_kwargs={
        "scope": "openid profile email",
        "code_challenge_method": "S256"
    }
)

print("Second Flask application initialized!")
print("SSO OAuth configuration completed!")

Second Flask application initialized!
SSO OAuth configuration completed!


In [16]:
@sso_app.route("/")
def sso_home():

    if session.get("sso_user"):
        return f"""
        <h1>SecureNova AI SSO Demo</h1>
        <h3>Welcome, {escape(session["sso_user"]["name"])}</h3>
        <p>Second application authenticated successfully!</p>
        <a href="/logout">Logout</a>
        """

    return """
    <h1>SecureNova AI SSO Demo</h1>
    <p>Second application using Auth0 SSO</p>
    <a href="/login">Login with Auth0</a>
    """


@sso_app.route("/login")
def sso_login():

    return sso_oauth.auth0.authorize_redirect(
        "http://localhost:5001/callback"
    )


@sso_app.route("/callback")
def sso_callback():

    token = sso_oauth.auth0.authorize_access_token()

    user_info = token.get("userinfo")

    if not user_info:
        raise RuntimeError("Auth0 user information not received.")

    session["sso_user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "sub": user_info.get("sub", "")
    }

    return redirect(url_for("sso_home"))


@sso_app.route("/logout")
def sso_logout():

    session.clear()

    logout_url = (
        f"https://{AUTH0_DOMAIN}/v2/logout?"
        + urlencode({
            "client_id": SSO_CLIENT_ID,
            "returnTo": "http://localhost:5001/"
        })
    )

    return redirect(logout_url)


print("SSO routes registered successfully!")

SSO routes registered successfully!


In [17]:
sso_server = make_server("127.0.0.1", 5001, sso_app)

sso_thread = Thread(
    target=sso_server.serve_forever,
    daemon=True
)

sso_thread.start()

print("SecureNova SSO Demo server started!")
print("Open: http://localhost:5001")

SecureNova SSO Demo server started!
Open: http://localhost:5001


127.0.0.1 - - [10/Nov/2026 18:25:16] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 18:25:18] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 18:25:20] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 18:25:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 18:25:23] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 18:25:53] "GET /callback?code=f1gCB7ARWF2O82ET3XxXiE9Rg8ok4SXsvsvoRTJKlyEPN&state=t3k8T2ajp6KdPetExEG5HJ6bgdbT1G&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 18:25:53] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 18:26:19] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 18:26:19] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [10/Nov/2026 18:26:23] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 18:26:37] "GET /callback?code=1kop4is8sR3mTdoHnL9THBNUX8Li5bPWyc2wXxW2QFQVA&state=bXVpsKtE9sIQ3ALZ6T5Yy0h02k7nKJ&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1

In [24]:
M2M_CLIENT_ID = input(
    "Enter SecureNova AI Agent Client ID: "
).strip()

M2M_CLIENT_SECRET = getpass(
    "Enter SecureNova AI Agent Client Secret: "
)

AGENT_API_AUDIENCE = "https://api-agent.securenova.com"

print("M2M credentials loaded successfully!")
print("Client ID loaded:", bool(M2M_CLIENT_ID))
print("Client Secret loaded:", bool(M2M_CLIENT_SECRET))

Enter SecureNova AI Agent Client ID:  i4qH92hXtscOn2q6FIt8f1FHE021Ml3t
Enter SecureNova AI Agent Client Secret:  ········


M2M credentials loaded successfully!
Client ID loaded: True
Client Secret loaded: True


In [28]:
token_response = requests.post(
    f"https://{AUTH0_DOMAIN}/oauth/token",
    json={
        "client_id": M2M_CLIENT_ID,
        "client_secret": M2M_CLIENT_SECRET,
        "audience": AGENT_API_AUDIENCE,
        "grant_type": "client_credentials",
        "scope": "read:ai-data"
    },
    timeout=30
)

print("HTTP Status:", token_response.status_code)

if token_response.status_code == 200:
    m2m_token_data = token_response.json()

    m2m_access_token = m2m_token_data["access_token"]

    print("M2M access token received successfully!")
    print("Token type:", m2m_token_data.get("token_type"))
    print("Expires in:", m2m_token_data.get("expires_in"), "seconds")
    print("Scope:", m2m_token_data.get("scope"))
else:
    print("Token request failed.")
    print("Response:", token_response.text)

HTTP Status: 200
M2M access token received successfully!
Token type: Bearer
Expires in: 60 seconds
Scope: read:ai-data


In [29]:
agent_signing_key = jwks_client.get_signing_key_from_jwt(
    m2m_access_token
)

agent_claims = jwt.decode(
    m2m_access_token,
    agent_signing_key.key,
    algorithms=["RS256"],
    audience=AGENT_API_AUDIENCE,
    issuer=f"https://{AUTH0_DOMAIN}/"
)

print("M2M JWT signature verified!")
print("Issuer:", agent_claims["iss"])
print("Audience:", agent_claims["aud"])
print("Expiry:", agent_claims["exp"])
print("Issued at:", agent_claims["iat"])
print("Subject:", agent_claims["sub"])
print("M2M token verification successful!")

M2M JWT signature verified!
Issuer: https://dev-wuaza68d7rbydra8.us.auth0.com/
Audience: https://api-agent.securenova.com
Expiry: 1794316416
Issued at: 1794316356
Subject: i4qH92hXtscOn2q6FIt8f1FHE021Ml3t@clients
M2M token verification successful!


In [30]:
from flask import request

agent_app = Flask("SecureNovaAgentAPI")

agent_app.config.update(
    SESSION_COOKIE_HTTPONLY=True,
    SESSION_COOKIE_SAMESITE="Lax"
)

print("SecureNova AI Agent API initialized!")

SecureNova AI Agent API initialized!


In [31]:
def verify_agent_token(token):

    signing_key = jwks_client.get_signing_key_from_jwt(token)

    claims = jwt.decode(
        token,
        signing_key.key,
        algorithms=["RS256"],
        audience=AGENT_API_AUDIENCE,
        issuer=f"https://{AUTH0_DOMAIN}/"
    )

    required_scope = "read:ai-data"

    token_scopes = claims.get("scope", "").split()

    if required_scope not in token_scopes:
        raise PermissionError("Required scope is missing.")

    return claims


print("Agent JWT verification function created!")

Agent JWT verification function created!


In [32]:
@agent_app.route("/chat", methods=["POST"])
def agent_chat():

    auth_header = request.headers.get("Authorization", "")

    if not auth_header.startswith("Bearer "):
        return jsonify({
            "error": "Missing or invalid Authorization header"
        }), 401

    token = auth_header.split(" ", 1)[1]

    try:
        claims = verify_agent_token(token)

    except PermissionError:
        return jsonify({
            "error": "Insufficient permissions"
        }), 403

    except Exception:
        return jsonify({
            "error": "Invalid or expired access token"
        }), 401

    data = request.get_json(silent=True) or {}

    message = data.get("message", "")

    return jsonify({
        "status": "success",
        "agent": "SecureNova AI Agent",
        "message_received": message,
        "authenticated_client": claims.get("sub"),
        "message": "Authorized access to SecureNova AI Agent API"
    }), 200


print("Protected /chat endpoint created successfully!")

Protected /chat endpoint created successfully!


In [33]:
agent_server = make_server("127.0.0.1", 5002, agent_app)

agent_thread = Thread(
    target=agent_server.serve_forever,
    daemon=True
)

agent_thread.start()

print("SecureNova AI Agent API started!")
print("Endpoint: http://localhost:5002/chat")

SecureNova AI Agent API started!
Endpoint: http://localhost:5002/chat


In [34]:
chat_response = requests.post(
    "http://localhost:5002/chat",
    headers={
        "Authorization": f"Bearer {m2m_access_token}"
    },
    json={
        "message": "Hello SecureNova AI Agent!"
    },
    timeout=30
)

print("HTTP Status:", chat_response.status_code)
print("Response:", chat_response.json())

127.0.0.1 - - [10/Nov/2026 18:46:41] "POST /chat HTTP/1.1" 401 -


HTTP Status: 401
Response: {'error': 'Invalid or expired access token'}


In [36]:
token_response = requests.post(
    f"https://{AUTH0_DOMAIN}/oauth/token",
    json={
        "client_id": M2M_CLIENT_ID,
        "client_secret": M2M_CLIENT_SECRET,
        "audience": AGENT_API_AUDIENCE,
        "grant_type": "client_credentials",
        "scope": "read:ai-data"
    },
    timeout=30
)

print("HTTP Status:", token_response.status_code)

if token_response.status_code == 200:
    m2m_token_data = token_response.json()
    m2m_access_token = m2m_token_data["access_token"]

    print("Fresh token received!")
    print("Expires in:", m2m_token_data.get("expires_in"))
    print("Scope:", m2m_token_data.get("scope"))
else:
    print(token_response.text)

HTTP Status: 200
Fresh token received!
Expires in: 60
Scope: read:ai-data


In [37]:
agent_signing_key = jwks_client.get_signing_key_from_jwt(
    m2m_access_token
)

agent_claims = jwt.decode(
    m2m_access_token,
    agent_signing_key.key,
    algorithms=["RS256"],
    options={"verify_signature": True, "verify_aud": False, "verify_exp": False},
    issuer=f"https://{AUTH0_DOMAIN}/"
)

print("Issuer:", agent_claims.get("iss"))
print("Audience:", agent_claims.get("aud"))
print("Scope:", agent_claims.get("scope"))
print("Expiry:", agent_claims.get("exp"))
print("Issued at:", agent_claims.get("iat"))

Issuer: https://dev-wuaza68d7rbydra8.us.auth0.com/
Audience: https://api-agent.securenova.com
Scope: read:ai-data
Expiry: 1794316773
Issued at: 1794316713


In [38]:
chat_response = requests.post(
    "http://localhost:5002/chat",
    headers={
        "Authorization": f"Bearer {m2m_access_token}"
    },
    json={
        "message": "Hello SecureNova AI Agent!"
    },
    timeout=30
)

print("HTTP Status:", chat_response.status_code)
print("Response:", chat_response.json())

127.0.0.1 - - [10/Nov/2026 18:48:56] "POST /chat HTTP/1.1" 200 -


HTTP Status: 200
Response: {'agent': 'SecureNova AI Agent', 'authenticated_client': 'i4qH92hXtscOn2q6FIt8f1FHE021Ml3t@clients', 'message': 'Authorized access to SecureNova AI Agent API', 'message_received': 'Hello SecureNova AI Agent!', 'status': 'success'}


In [39]:
import time

print("Waiting 65 seconds for token expiration...")
time.sleep(65)
print("Token expiration wait completed.")

Waiting 65 seconds for token expiration...
Token expiration wait completed.


In [40]:
expired_response = requests.post(
    "http://localhost:5002/chat",
    headers={
        "Authorization": f"Bearer {m2m_access_token}"
    },
    json={
        "message": "Replay attempt after token expiry"
    },
    timeout=30
)

print("HTTP Status:", expired_response.status_code)
print("Response:", expired_response.json())

127.0.0.1 - - [10/Nov/2026 18:50:41] "POST /chat HTTP/1.1" 401 -


HTTP Status: 401
Response: {'error': 'Invalid or expired access token'}


In [41]:
fresh_response = requests.post(
    "http://localhost:5002/chat",
    headers={
        "Authorization": f"Bearer {m2m_access_token}"
    },
    json={
        "message": "Testing with a newly issued token"
    },
    timeout=30
)

print("HTTP Status:", fresh_response.status_code)
print("Response:", fresh_response.json())

127.0.0.1 - - [10/Nov/2026 18:50:56] "POST /chat HTTP/1.1" 401 -


HTTP Status: 401
Response: {'error': 'Invalid or expired access token'}


In [42]:
token_response = requests.post(
    f"https://{AUTH0_DOMAIN}/oauth/token",
    json={
        "client_id": M2M_CLIENT_ID,
        "client_secret": M2M_CLIENT_SECRET,
        "audience": AGENT_API_AUDIENCE,
        "grant_type": "client_credentials",
        "scope": "read:ai-data"
    },
    timeout=30
)

if token_response.status_code == 200:
    m2m_token_data = token_response.json()
    m2m_access_token = m2m_token_data["access_token"]

    print("Fresh token generated!")
    print("Expires in:", m2m_token_data.get("expires_in"), "seconds")
else:
    print("Token error:", token_response.text)

Fresh token generated!
Expires in: 60 seconds


In [43]:
fresh_response = requests.post(
    "http://localhost:5002/chat",
    headers={
        "Authorization": f"Bearer {m2m_access_token}"
    },
    json={
        "message": "Testing with a newly issued token"
    },
    timeout=30
)

print("HTTP Status:", fresh_response.status_code)
print("Response:", fresh_response.json())

127.0.0.1 - - [10/Nov/2026 18:52:15] "POST /chat HTTP/1.1" 200 -


HTTP Status: 200
Response: {'agent': 'SecureNova AI Agent', 'authenticated_client': 'i4qH92hXtscOn2q6FIt8f1FHE021Ml3t@clients', 'message': 'Authorized access to SecureNova AI Agent API', 'message_received': 'Testing with a newly issued token', 'status': 'success'}


In [44]:
no_token_response = requests.post(
    "http://localhost:5002/chat",
    json={"message": "Unauthorized request"},
    timeout=30
)

print("No-token test:", no_token_response.status_code)
print(no_token_response.json())

127.0.0.1 - - [10/Nov/2026 18:52:54] "POST /chat HTTP/1.1" 401 -


No-token test: 401
{'error': 'Missing or invalid Authorization header'}


In [45]:
user_access_token = token_store[latest_token_session_id]

wrong_audience_response = requests.post(
    "http://localhost:5002/chat",
    headers={
        "Authorization": f"Bearer {user_access_token}"
    },
    json={"message": "Testing wrong audience"},
    timeout=30
)

print("Wrong-audience test:", wrong_audience_response.status_code)
print(wrong_audience_response.json())

127.0.0.1 - - [10/Nov/2026 18:53:04] "POST /chat HTTP/1.1" 401 -


Wrong-audience test: 401
{'error': 'Invalid or expired access token'}


In [46]:
admin_response = requests.get(
    "http://localhost:5003/api/admin",
    headers={
        "Authorization": f"Bearer {user_access_token}"
    },
    timeout=30
)

print("Admin endpoint status:", admin_response.status_code)
print("Response:", admin_response.json())

127.0.0.1 - - [10/Nov/2026 18:53:14] "GET /api/admin HTTP/1.1" 401 -


Admin endpoint status: 401
Response: {'error': 'Invalid or expired access token'}
